# CUSUM 기반 분진 폭발 예측 시스템
## 슈레더 공정 분진 농도 모니터링 및 위험 예측

**목적:** 슈레더(파쇄기) 공정에서 발생하는 금속 분진의 농도를 CUSUM(누적합) 알고리즘으로 모니터링하여,
분진 폭발 위험을 사전에 예측하고 경보를 발생시키는 시스템

**핵심 기술:**
- **CUSUM (Cumulative Sum)**: 미세한 평균 이동을 조기에 탐지하는 통계적 공정 관리 기법
- **시간-위험 예측**: 현재 추세를 외삽하여 폭발 하한 농도(LEL) 도달 시점 예측
- **복합 위험 점수**: 농도, CUSUM 수준, 변화율을 종합한 0~1 위험 지수

**법적 기준:**
- TWA-8hr (시간가중평균): 10 mg/m³ (산업안전보건법)
- LEL (폭발 하한 농도): 40 mg/m³ (금속 분진 기준)

**데이터:** 30일간 60초 간격 측정 (43,200 샘플)

## Step 0. 라이브러리 임포트

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

# Matplotlib settings
plt.rcParams.update({
    'figure.figsize': (16, 6),
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

print("Libraries loaded successfully.")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## Step 1. 분진 센서 데이터 생성 (30일)

DST-1 레이저 분진 센서 시뮬레이션:
- **측정 주기:** 60초 간격, 총 43,200 샘플
- **기본 농도:** 가동 중 5.0 mg/m³, 비가동 시 0.5 mg/m³
- **가동 시간:** 평일(월~금) 08:00~18:00
- **이벤트 유형:** 점진적 상승(drift), 급격한 스파이크, 진동(oscillation), 지속적 고농도, 회복 패턴
- **마모 트렌드:** 시간 경과에 따른 기저 농도 점진적 증가

In [ ]:
np.random.seed(42)

# --- Time index: 30 days, 60-second intervals ---
n_days = 30
freq_sec = 60
start_time = pd.Timestamp('2026-03-01 00:00:00')
n_samples = n_days * 24 * 60  # 43,200
timestamps = pd.date_range(start=start_time, periods=n_samples, freq=f'{freq_sec}s')

# --- Operating mask: weekday 08:00-18:00 ---
hours = timestamps.hour
weekdays = timestamps.weekday  # 0=Mon, 6=Sun
operating = (weekdays < 5) & (hours >= 8) & (hours < 18)

# --- Base concentration ---
base_level_on = 5.0   # mg/m³ during operation
base_level_off = 0.5  # mg/m³ during shutdown
concentration = np.where(operating, base_level_on, base_level_off)

# --- Natural variation (Gaussian noise) ---
noise = np.where(operating, np.random.normal(0, 0.8, n_samples),
                 np.random.normal(0, 0.1, n_samples))
concentration = concentration + noise

# --- Wear trend: gradual baseline increase over 30 days ---
day_fraction = np.arange(n_samples) / n_samples
wear_trend = 1.5 * day_fraction * operating  # up to +1.5 mg/m³ by day 30
concentration = concentration + wear_trend

# --- Event definitions ---
events = []

# Event 1: Gradual drift (Day 5-6, filter clogging)
e1_start = pd.Timestamp('2026-03-06 09:00')
e1_end = pd.Timestamp('2026-03-06 16:00')
mask1 = (timestamps >= e1_start) & (timestamps <= e1_end) & operating
drift_vals = np.linspace(0, 6.0, mask1.sum())
concentration[mask1] += drift_vals
events.append({'name': 'Gradual Drift (Filter Clog)', 'start': e1_start, 'end': e1_end, 'type': 'drift'})

# Event 2: Sudden spike (Day 10, foreign material)
e2_start = pd.Timestamp('2026-03-11 11:00')
e2_end = pd.Timestamp('2026-03-11 11:30')
mask2 = (timestamps >= e2_start) & (timestamps <= e2_end)
spike_profile = 30 * np.exp(-np.linspace(0, 4, mask2.sum()))
concentration[mask2] += spike_profile
events.append({'name': 'Sudden Spike (Foreign Material)', 'start': e2_start, 'end': e2_end, 'type': 'spike'})

# Event 3: Oscillation (Day 14-15, unstable ventilation)
e3_start = pd.Timestamp('2026-03-15 08:00')
e3_end = pd.Timestamp('2026-03-16 18:00')
mask3 = (timestamps >= e3_start) & (timestamps <= e3_end) & operating
t3 = np.arange(mask3.sum())
oscillation = 4.0 * np.sin(2 * np.pi * t3 / 120) * np.exp(-t3 / 800)
concentration[mask3] += np.abs(oscillation)
events.append({'name': 'Oscillation (Ventilation Issue)', 'start': e3_start, 'end': e3_end, 'type': 'oscillation'})

# Event 4: Sustained high (Day 20-21, blade wear + overload)
e4_start = pd.Timestamp('2026-03-21 08:00')
e4_end = pd.Timestamp('2026-03-22 18:00')
mask4 = (timestamps >= e4_start) & (timestamps <= e4_end) & operating
concentration[mask4] += 7.0 + np.random.normal(0, 1.0, mask4.sum())
events.append({'name': 'Sustained High (Blade Wear)', 'start': e4_start, 'end': e4_end, 'type': 'sustained'})

# Event 5: Recovery after maintenance (Day 23)
e5_start = pd.Timestamp('2026-03-23 08:00')
e5_end = pd.Timestamp('2026-03-23 14:00')
mask5 = (timestamps >= e5_start) & (timestamps <= e5_end) & operating
recovery = np.linspace(5.0, 0, mask5.sum())
concentration[mask5] += recovery
events.append({'name': 'Recovery (Post-Maintenance)', 'start': e5_start, 'end': e5_end, 'type': 'recovery'})

# Event 6: Critical drift (Day 27-28, progressive failure)
e6_start = pd.Timestamp('2026-03-27 08:00')
e6_end = pd.Timestamp('2026-03-28 18:00')
mask6 = (timestamps >= e6_start) & (timestamps <= e6_end) & operating
progressive = np.linspace(0, 15.0, mask6.sum()) + np.random.normal(0, 1.5, mask6.sum())
concentration[mask6] += progressive
events.append({'name': 'Critical Drift (Progressive Failure)', 'start': e6_start, 'end': e6_end, 'type': 'drift'})

# Event 7: Short spike (Day 18)
e7_start = pd.Timestamp('2026-03-18 14:00')
e7_end = pd.Timestamp('2026-03-18 14:15')
mask7 = (timestamps >= e7_start) & (timestamps <= e7_end)
concentration[mask7] += 20 * np.exp(-np.linspace(0, 3, mask7.sum()))
events.append({'name': 'Short Spike (Material Jam)', 'start': e7_start, 'end': e7_end, 'type': 'spike'})

# --- Ensure non-negative ---
concentration = np.clip(concentration, 0, None)

# --- Build DataFrame ---
df = pd.DataFrame({
    'timestamp': timestamps,
    'dust_mg_m3': concentration,
    'operating': operating,
})
df.set_index('timestamp', inplace=True)

# --- Legal limits ---
TWA_LIMIT = 10.0   # mg/m³ (8-hour TWA)
LEL_LIMIT = 40.0   # mg/m³ (Lower Explosive Limit, metal dust)

print(f"데이터 생성 완료: {len(df):,} 샘플 ({n_days}일)")
print(f"기간: {df.index[0]} ~ {df.index[-1]}")
print(f"\n기본 통계 (가동 시간 중):")
op_data = df[df['operating']]['dust_mg_m3']
print(f"  평균: {op_data.mean():.2f} mg/m³")
print(f"  최대: {op_data.max():.2f} mg/m³")
print(f"  TWA 초과 비율: {(op_data > TWA_LIMIT).mean()*100:.1f}%")
print(f"  LEL 초과 비율: {(op_data > LEL_LIMIT).mean()*100:.2f}%")
print(f"\n이벤트 수: {len(events)}건")

## Step 2. 데이터 시각화

30일간의 분진 농도 데이터를 개관합니다. 일별/주별 패턴과 주요 이벤트를 확인합니다.

In [ ]:
# 2-1. 30-day overview
fig, ax = plt.subplots(figsize=(18, 6))
ax.plot(df.index, df['dust_mg_m3'], linewidth=0.3, color='steelblue', alpha=0.7)
ax.axhline(y=TWA_LIMIT, color='orange', linestyle='--', linewidth=1.5, label=f'TWA-8hr Limit ({TWA_LIMIT} mg/m3)')
ax.axhline(y=LEL_LIMIT, color='red', linestyle='--', linewidth=1.5, label=f'LEL Metal Dust ({LEL_LIMIT} mg/m3)')

# Mark events
event_colors = {'drift': 'orange', 'spike': 'red', 'oscillation': 'purple',
                'sustained': 'brown', 'recovery': 'green'}
for ev in events:
    ax.axvspan(ev['start'], ev['end'], alpha=0.15, color=event_colors[ev['type']])

ax.set_xlabel('Date')
ax.set_ylabel('Dust Concentration (mg/m3)')
ax.set_title('DST-1 Laser Dust Sensor - 30 Day Overview (Shredder Process)')
ax.legend(loc='upper left')
ax.set_xlim(df.index[0], df.index[-1])
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 2-2. Daily pattern (hourly average) and Weekly pattern
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Hourly pattern
op_df = df[df['operating']].copy()
op_df['hour'] = op_df.index.hour
hourly = op_df.groupby('hour')['dust_mg_m3'].agg(['mean', 'std'])
axes[0].bar(hourly.index, hourly['mean'], color='steelblue', alpha=0.7, yerr=hourly['std'], capsize=3)
axes[0].axhline(y=TWA_LIMIT, color='orange', linestyle='--', label='TWA Limit')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Dust Concentration (mg/m3)')
axes[0].set_title('Hourly Dust Pattern (Operating Hours)')
axes[0].legend()

# Weekly pattern
op_df['weekday'] = op_df.index.weekday
daily_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
weekly = op_df.groupby('weekday')['dust_mg_m3'].agg(['mean', 'std'])
weekly = weekly.loc[:4]  # weekdays only
axes[1].bar(range(5), weekly['mean'], color='coral', alpha=0.7, yerr=weekly['std'], capsize=3)
axes[1].set_xticks(range(5))
axes[1].set_xticklabels(daily_names)
axes[1].axhline(y=TWA_LIMIT, color='orange', linestyle='--', label='TWA Limit')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Dust Concentration (mg/m3)')
axes[1].set_title('Weekly Dust Pattern')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 2-3. Event zoom-ins (3 key events)
zoom_events = [
    ('Sudden Spike - Foreign Material (Mar 11)', '2026-03-11 10:30', '2026-03-11 12:30'),
    ('Sustained High - Blade Wear (Mar 21-22)', '2026-03-21 06:00', '2026-03-22 20:00'),
    ('Critical Drift - Progressive Failure (Mar 27-28)', '2026-03-27 06:00', '2026-03-28 20:00'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (title, zs, ze) in zip(axes, zoom_events):
    mask = (df.index >= zs) & (df.index <= ze)
    ax.plot(df.index[mask], df['dust_mg_m3'][mask], linewidth=0.8, color='steelblue')
    ax.axhline(y=TWA_LIMIT, color='orange', linestyle='--', linewidth=1, label='TWA')
    ax.axhline(y=LEL_LIMIT, color='red', linestyle='--', linewidth=1, label='LEL')
    ax.set_title(title, fontsize=10)
    ax.set_ylabel('mg/m3')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.legend(fontsize=8)

plt.suptitle('Event Zoom-In Details', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Step 3. CUSUM 알고리즘

**CUSUM (Cumulative Sum) 제어 차트** 는 공정 평균의 미세한 변화를 조기에 감지하는 기법입니다.

**수식:**
- 상향 CUSUM: $S^+(n) = \max(0,\ S^+(n-1) + (x_n - \mu_0 - k))$
- 하향 CUSUM: $S^-(n) = \max(0,\ S^-(n-1) - (x_n - \mu_0 + k))$

**파라미터:**
- $\mu_0$: 목표 평균 (정상 가동 시 기준값)
- $k$: 허용 여유 (allowance) - 감지 민감도 결정
- $h$: 결정 구간 (decision interval) - 경보 임계값

**보정:** 첫 24시간(가동 시간)의 안정적인 데이터에서 $\mu_0$와 $\sigma$를 추정합니다.

In [ ]:
# --- CUSUM Parameter Calibration from first 24h of stable operating data ---
cal_end = start_time + pd.Timedelta(hours=24)
cal_data = df[(df.index < cal_end) & (df['operating'])]['dust_mg_m3']

mu_0 = cal_data.mean()   # target mean
sigma = cal_data.std()    # process std

# CUSUM parameters (standard: k = 0.5*sigma, h = 5*sigma)
k = 0.5 * sigma   # allowance (half-sigma shift detection)
h = 5.0 * sigma    # decision interval (alarm threshold)

print(f"CUSUM 보정 결과 (첫 24시간 가동 데이터 기준):")
print(f"  mu_0 (목표 평균): {mu_0:.3f} mg/m3")
print(f"  sigma (공정 표준편차): {sigma:.3f} mg/m3")
print(f"  k (허용 여유): {k:.3f} mg/m3")
print(f"  h (결정 구간): {h:.3f} mg/m3")
print(f"  보정 샘플 수: {len(cal_data)}")

# --- Compute CUSUM (operating hours only, carry forward during non-op) ---
x = df['dust_mg_m3'].values
op = df['operating'].values

S_plus = np.zeros(n_samples)   # upper CUSUM
S_minus = np.zeros(n_samples)  # lower CUSUM
alarm_upper = np.zeros(n_samples, dtype=bool)
alarm_lower = np.zeros(n_samples, dtype=bool)

for i in range(1, n_samples):
    if op[i]:
        S_plus[i] = max(0, S_plus[i-1] + (x[i] - mu_0 - k))
        S_minus[i] = max(0, S_minus[i-1] - (x[i] - mu_0 + k))
    else:
        # Decay during non-operating hours (gradual reset)
        S_plus[i] = S_plus[i-1] * 0.995
        S_minus[i] = S_minus[i-1] * 0.995

    alarm_upper[i] = S_plus[i] > h
    alarm_lower[i] = S_minus[i] > h

df['cusum_plus'] = S_plus
df['cusum_minus'] = S_minus
df['alarm_upper'] = alarm_upper
df['alarm_lower'] = alarm_lower

n_upper_alarms = alarm_upper.sum()
n_lower_alarms = alarm_lower.sum()
print(f"\nCUSUM 결과:")
print(f"  상향 경보 포인트: {n_upper_alarms:,}개 ({n_upper_alarms/n_samples*100:.1f}%)")
print(f"  하향 경보 포인트: {n_lower_alarms:,}개")
print(f"  S+ 최대값: {S_plus.max():.2f}")
print(f"  S- 최대값: {S_minus.max():.2f}")

## Step 4. 위험 도달 시간 예측 (Time-to-Danger)

CUSUM이 이상을 감지했을 때, 현재 농도 추세를 선형 외삽하여 LEL(40 mg/m³) 도달 예상 시간을 계산합니다.

- 최근 30분(30포인트)의 이동 평균 기울기를 사용
- 기울기가 양수일 때만 도달 시간 계산 (농도가 상승 중)
- 95% 신뢰 구간 포함

In [ ]:
# --- Time-to-Danger Prediction ---
window = 30  # 30 minutes rolling window

# Rolling slope (mg/m3 per minute)
dust = df['dust_mg_m3'].values
rolling_slope = np.full(n_samples, np.nan)
rolling_slope_std = np.full(n_samples, np.nan)

for i in range(window, n_samples):
    if op[i]:
        segment = dust[i-window:i]
        t_local = np.arange(window)
        if np.std(segment) > 0.01:
            coeffs = np.polyfit(t_local, segment, 1)
            residuals = segment - np.polyval(coeffs, t_local)
            rolling_slope[i] = coeffs[0]  # mg/m3 per minute
            rolling_slope_std[i] = np.std(residuals)

df['slope'] = rolling_slope  # mg/m3 per minute

# Time to LEL (minutes)
time_to_lel = np.full(n_samples, np.nan)
time_to_lel_lower = np.full(n_samples, np.nan)  # conservative (fast arrival)
time_to_lel_upper = np.full(n_samples, np.nan)  # optimistic (slow arrival)

for i in range(window, n_samples):
    if op[i] and not np.isnan(rolling_slope[i]) and rolling_slope[i] > 0.01:
        remaining = LEL_LIMIT - dust[i]
        if remaining > 0:
            ttl = remaining / rolling_slope[i]  # minutes
            time_to_lel[i] = ttl
            # 95% CI using slope uncertainty
            slope_upper = rolling_slope[i] + 1.96 * rolling_slope_std[i] / np.sqrt(window)
            slope_lower = max(rolling_slope[i] - 1.96 * rolling_slope_std[i] / np.sqrt(window), 0.001)
            time_to_lel_lower[i] = remaining / slope_upper if slope_upper > 0 else np.nan
            time_to_lel_upper[i] = remaining / slope_lower

df['time_to_lel_min'] = time_to_lel
df['time_to_lel_lower'] = time_to_lel_lower
df['time_to_lel_upper'] = time_to_lel_upper

# Convert to hours for readability
df['time_to_lel_hr'] = df['time_to_lel_min'] / 60

# Stats on alarm periods
alarm_ttl = df[df['alarm_upper'] & ~df['time_to_lel_hr'].isna()]['time_to_lel_hr']
print(f"위험 도달 시간 예측 결과 (CUSUM 경보 발생 구간):")
if len(alarm_ttl) > 0:
    print(f"  예측 가능 포인트: {len(alarm_ttl):,}개")
    print(f"  최소 도달 시간: {alarm_ttl.min():.1f} 시간")
    print(f"  평균 도달 시간: {alarm_ttl.mean():.1f} 시간")
    print(f"  중앙값 도달 시간: {alarm_ttl.median():.1f} 시간")
else:
    print("  경보 구간에서 예측 가능 포인트 없음")

## Step 5. 폭발 위험 점수 (Explosion Risk Score)

3가지 요소를 결합한 복합 위험 점수 (0~1):

| 요소 | 가중치 | 설명 |
|------|--------|------|
| 농도 비율 | 40% | 현재 농도 / LEL |
| CUSUM 수준 | 35% | S+ / (3 * h) |
| 변화율 | 25% | 기울기 정규화 |

**위험 등급:**
- **안전 (Safe):** < 0.3
- **주의 (Caution):** 0.3 ~ 0.6
- **경고 (Warning):** 0.6 ~ 0.8
- **위험 (Danger):** > 0.8

In [ ]:
# --- Explosion Risk Score ---
# Component 1: Concentration ratio (0~1)
conc_ratio = np.clip(df['dust_mg_m3'].values / LEL_LIMIT, 0, 1)

# Component 2: CUSUM level (0~1)
cusum_ratio = np.clip(df['cusum_plus'].values / (3 * h), 0, 1)

# Component 3: Rate of change (0~1)
slope_vals = df['slope'].fillna(0).values
# Normalize: 0.5 mg/m3/min = max concern
roc_ratio = np.clip(slope_vals / 0.5, 0, 1)

# Weighted composite
w_conc, w_cusum, w_roc = 0.40, 0.35, 0.25
risk_score = w_conc * conc_ratio + w_cusum * cusum_ratio + w_roc * roc_ratio
risk_score = np.clip(risk_score, 0, 1)

# Zero out non-operating periods (keep minimal risk)
risk_score = np.where(op, risk_score, risk_score * 0.1)

df['risk_score'] = risk_score

# Risk zone classification
def classify_risk(score):
    if score < 0.3:
        return 'Safe'
    elif score < 0.6:
        return 'Caution'
    elif score < 0.8:
        return 'Warning'
    else:
        return 'Danger'

df['risk_zone'] = df['risk_score'].apply(classify_risk)

# Stats
zone_counts = df[df['operating']]['risk_zone'].value_counts()
print("위험 등급 분포 (가동 시간 중):")
for zone in ['Safe', 'Caution', 'Warning', 'Danger']:
    count = zone_counts.get(zone, 0)
    pct = count / len(df[df['operating']]) * 100
    print(f"  {zone}: {count:,}개 ({pct:.1f}%)")

print(f"\n최대 위험 점수: {risk_score[op].max():.3f}")
print(f"평균 위험 점수 (가동 중): {risk_score[op].mean():.3f}")

## Step 6. 종합 시각화

각 차트를 개별 셀에서 생성합니다.

### 6-1. 30일 분진 농도 + 경보선

In [ ]:
# 6-1. 30-day dust concentration with alert lines
fig, ax = plt.subplots(figsize=(18, 6))

# Color by risk zone
colors_map = {'Safe': 'steelblue', 'Caution': 'goldenrod', 'Warning': 'orange', 'Danger': 'red'}
for zone, color in colors_map.items():
    mask = df['risk_zone'] == zone
    if mask.any():
        ax.scatter(df.index[mask], df['dust_mg_m3'][mask], s=0.3, c=color, alpha=0.5, label=zone)

ax.axhline(y=TWA_LIMIT, color='orange', linestyle='--', linewidth=2, label=f'TWA-8hr ({TWA_LIMIT} mg/m3)')
ax.axhline(y=LEL_LIMIT, color='red', linestyle='-', linewidth=2, label=f'LEL ({LEL_LIMIT} mg/m3)')

# Shade alarm regions
alarm_starts = []
in_alarm = False
for i in range(len(df)):
    if alarm_upper[i] and not in_alarm:
        alarm_starts.append(i)
        in_alarm = True
    elif not alarm_upper[i] and in_alarm:
        ax.axvspan(df.index[alarm_starts[-1]], df.index[i], alpha=0.1, color='red')
        in_alarm = False
if in_alarm:
    ax.axvspan(df.index[alarm_starts[-1]], df.index[-1], alpha=0.1, color='red')

ax.set_xlabel('Date')
ax.set_ylabel('Dust Concentration (mg/m3)')
ax.set_title('30-Day Dust Concentration Monitoring - DST-1 Sensor (Shredder)')
ax.legend(loc='upper left', markerscale=10, fontsize=9)
ax.set_xlim(df.index[0], df.index[-1])
ax.set_ylim(bottom=0)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 6-2. CUSUM 제어 차트 (S+ / S-)

In [ ]:
# 6-2. CUSUM+ / CUSUM- control chart
fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

# Upper CUSUM
ax1 = axes[0]
ax1.plot(df.index, df['cusum_plus'], linewidth=0.5, color='steelblue', label='S+ (Upper CUSUM)')
ax1.axhline(y=h, color='red', linestyle='--', linewidth=1.5, label=f'Decision Interval h={h:.2f}')
ax1.fill_between(df.index, 0, df['cusum_plus'],
                  where=df['cusum_plus'] > h, color='red', alpha=0.3, label='Alarm Zone')
ax1.set_ylabel('S+ Value')
ax1.set_title('Upper CUSUM (S+) - Detects Upward Drift')
ax1.legend(loc='upper left')
ax1.set_ylim(bottom=0)

# Lower CUSUM
ax2 = axes[1]
ax2.plot(df.index, df['cusum_minus'], linewidth=0.5, color='teal', label='S- (Lower CUSUM)')
ax2.axhline(y=h, color='red', linestyle='--', linewidth=1.5, label=f'Decision Interval h={h:.2f}')
ax2.fill_between(df.index, 0, df['cusum_minus'],
                  where=df['cusum_minus'] > h, color='red', alpha=0.3, label='Alarm Zone')
ax2.set_xlabel('Date')
ax2.set_ylabel('S- Value')
ax2.set_title('Lower CUSUM (S-) - Detects Downward Drift')
ax2.legend(loc='upper left')
ax2.set_ylim(bottom=0)

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax2.xaxis.set_major_locator(mdates.DayLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 6-3. 폭발 위험 점수 (색상별 위험 구간)

In [ ]:
# 6-3. Explosion risk score with color-coded zones
fig, ax = plt.subplots(figsize=(18, 6))

# Background zones
ax.axhspan(0, 0.3, alpha=0.1, color='green', label='Safe (< 0.3)')
ax.axhspan(0.3, 0.6, alpha=0.1, color='gold', label='Caution (0.3-0.6)')
ax.axhspan(0.6, 0.8, alpha=0.15, color='orange', label='Warning (0.6-0.8)')
ax.axhspan(0.8, 1.0, alpha=0.15, color='red', label='Danger (> 0.8)')

# Plot risk score
ax.plot(df.index, df['risk_score'], linewidth=0.4, color='black', alpha=0.7)

# Highlight danger zones
danger_mask = df['risk_score'] > 0.8
if danger_mask.any():
    ax.scatter(df.index[danger_mask], df['risk_score'][danger_mask],
              s=2, c='red', alpha=0.8, zorder=5)

ax.set_xlabel('Date')
ax.set_ylabel('Explosion Risk Score')
ax.set_title('Composite Explosion Risk Score (Concentration + CUSUM + Rate of Change)')
ax.set_ylim(0, 1.05)
ax.set_xlim(df.index[0], df.index[-1])
ax.legend(loc='upper left', fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 6-4. 위험 도달 시간 산점도 (Time-to-Danger)

In [ ]:
# 6-4. Time-to-danger scatter
fig, ax = plt.subplots(figsize=(18, 6))

# Only plot where prediction exists and is finite
valid = ~df['time_to_lel_hr'].isna() & (df['time_to_lel_hr'] < 500) & df['operating']
ttl_data = df[valid]

# Color by urgency
colors = np.where(ttl_data['time_to_lel_hr'] < 1, 'red',
         np.where(ttl_data['time_to_lel_hr'] < 4, 'orange',
         np.where(ttl_data['time_to_lel_hr'] < 12, 'gold', 'green')))

ax.scatter(ttl_data.index, ttl_data['time_to_lel_hr'], s=1.5, c=colors, alpha=0.6)

# Reference lines
ax.axhline(y=1, color='red', linestyle='--', linewidth=1, label='Critical: 1 hour')
ax.axhline(y=4, color='orange', linestyle='--', linewidth=1, label='Urgent: 4 hours')
ax.axhline(y=12, color='gold', linestyle='--', linewidth=1, label='Caution: 12 hours')

# Confidence interval for alarm periods
alarm_valid = valid & df['alarm_upper']
if alarm_valid.any():
    ax.fill_between(df.index[alarm_valid],
                    df['time_to_lel_lower'][alarm_valid] / 60,
                    np.minimum(df['time_to_lel_upper'][alarm_valid] / 60, 500),
                    alpha=0.15, color='red', label='95% CI (Alarm Period)')

ax.set_xlabel('Date')
ax.set_ylabel('Time to LEL (hours)')
ax.set_title('Predicted Time to Explosion Threshold (LEL = 40 mg/m3)')
ax.set_yscale('log')
ax.set_ylim(0.1, 500)
ax.set_xlim(df.index[0], df.index[-1])
ax.legend(loc='upper right', fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 6-5. 일별 TWA-8hr 준수 현황

In [ ]:
# 6-5. Daily TWA-8hr compliance check
op_df = df[df['operating']].copy()
op_df['date'] = op_df.index.date

# Calculate 8-hour TWA for each operating day
daily_twa = op_df.groupby('date')['dust_mg_m3'].mean()
daily_twa = daily_twa[daily_twa.index >= pd.Timestamp('2026-03-02').date()]  # skip partial first day

fig, ax = plt.subplots(figsize=(16, 5))

colors = ['green' if v <= TWA_LIMIT else 'red' for v in daily_twa.values]
bars = ax.bar(range(len(daily_twa)), daily_twa.values, color=colors, alpha=0.7, edgecolor='gray', linewidth=0.5)

ax.axhline(y=TWA_LIMIT, color='red', linestyle='--', linewidth=2, label=f'TWA-8hr Limit ({TWA_LIMIT} mg/m3)')
ax.set_xticks(range(len(daily_twa)))
ax.set_xticklabels([d.strftime('%m/%d') for d in daily_twa.index], rotation=45, fontsize=8)
ax.set_xlabel('Date')
ax.set_ylabel('TWA-8hr (mg/m3)')
ax.set_title('Daily TWA-8hr Compliance Check (Shredder Dust)')
ax.legend()

# Add pass/fail count
n_pass = sum(1 for v in daily_twa.values if v <= TWA_LIMIT)
n_fail = len(daily_twa) - n_pass
ax.text(0.98, 0.95, f'Pass: {n_pass} days / Fail: {n_fail} days',
        transform=ax.transAxes, ha='right', va='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

### 6-6. 이벤트 감지 타임라인

In [ ]:
# 6-6. Event detection timeline
fig, axes = plt.subplots(3, 1, figsize=(18, 10), sharex=True)

# Panel 1: Dust concentration
ax1 = axes[0]
ax1.plot(df.index, df['dust_mg_m3'], linewidth=0.3, color='steelblue', alpha=0.7)
ax1.axhline(y=TWA_LIMIT, color='orange', linestyle='--', linewidth=1)
ax1.axhline(y=LEL_LIMIT, color='red', linestyle='--', linewidth=1)
for ev in events:
    color = event_colors[ev['type']]
    ax1.axvspan(ev['start'], ev['end'], alpha=0.2, color=color)
    mid = ev['start'] + (ev['end'] - ev['start']) / 2
    ax1.annotate(ev['name'].split('(')[0].strip(), xy=(mid, ax1.get_ylim()[1] * 0.9),
                fontsize=7, ha='center', rotation=30, color=color, fontweight='bold')
ax1.set_ylabel('Dust (mg/m3)')
ax1.set_title('Event Detection Timeline - Concentration / CUSUM / Risk')

# Panel 2: CUSUM
ax2 = axes[1]
ax2.plot(df.index, df['cusum_plus'], linewidth=0.5, color='steelblue', label='S+')
ax2.plot(df.index, df['cusum_minus'], linewidth=0.5, color='teal', label='S-')
ax2.axhline(y=h, color='red', linestyle='--', linewidth=1, label='Threshold h')
for ev in events:
    ax2.axvspan(ev['start'], ev['end'], alpha=0.1, color=event_colors[ev['type']])
ax2.set_ylabel('CUSUM Value')
ax2.legend(loc='upper left', fontsize=8)

# Panel 3: Risk score
ax3 = axes[2]
ax3.fill_between(df.index, 0, df['risk_score'], alpha=0.5,
                  color='steelblue', where=df['risk_score'] < 0.3)
ax3.fill_between(df.index, 0, df['risk_score'], alpha=0.5,
                  color='gold', where=(df['risk_score'] >= 0.3) & (df['risk_score'] < 0.6))
ax3.fill_between(df.index, 0, df['risk_score'], alpha=0.5,
                  color='orange', where=(df['risk_score'] >= 0.6) & (df['risk_score'] < 0.8))
ax3.fill_between(df.index, 0, df['risk_score'], alpha=0.7,
                  color='red', where=df['risk_score'] >= 0.8)
for ev in events:
    ax3.axvspan(ev['start'], ev['end'], alpha=0.1, color=event_colors[ev['type']])
ax3.set_ylabel('Risk Score')
ax3.set_xlabel('Date')
ax3.set_ylim(0, 1.05)

ax3.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax3.xaxis.set_major_locator(mdates.DayLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 7. 종합 요약

### 분석 결과 요약

In [ ]:
# --- Summary Report ---
print("=" * 70)
print("  CUSUM 기반 분진 폭발 예측 시스템 - 종합 보고서")
print("=" * 70)

print(f"\n[데이터 개요]")
print(f"  분석 기간: {df.index[0].strftime('%Y-%m-%d')} ~ {df.index[-1].strftime('%Y-%m-%d')} ({n_days}일)")
print(f"  총 샘플 수: {n_samples:,}개 (60초 간격)")
print(f"  가동 시간 샘플: {op.sum():,}개 ({op.sum()/n_samples*100:.1f}%)")

print(f"\n[CUSUM 파라미터]")
print(f"  목표 평균 (mu_0): {mu_0:.3f} mg/m3")
print(f"  허용 여유 (k): {k:.3f} mg/m3")
print(f"  결정 구간 (h): {h:.3f} mg/m3")

print(f"\n[농도 통계 - 가동 시간]")
op_dust = df[df['operating']]['dust_mg_m3']
print(f"  평균: {op_dust.mean():.2f} mg/m3")
print(f"  표준편차: {op_dust.std():.2f} mg/m3")
print(f"  최대: {op_dust.max():.2f} mg/m3")
print(f"  TWA 초과: {(op_dust > TWA_LIMIT).sum():,}건 ({(op_dust > TWA_LIMIT).mean()*100:.1f}%)")
print(f"  LEL 초과: {(op_dust > LEL_LIMIT).sum():,}건 ({(op_dust > LEL_LIMIT).mean()*100:.2f}%)")

print(f"\n[CUSUM 경보]")
print(f"  상향 경보 포인트: {alarm_upper.sum():,}개")
print(f"  S+ 최대값: {S_plus.max():.1f}")

print(f"\n[위험 등급 분포 - 가동 시간]")
op_risk = df[df['operating']]
for zone in ['Safe', 'Caution', 'Warning', 'Danger']:
    cnt = (op_risk['risk_zone'] == zone).sum()
    pct = cnt / len(op_risk) * 100
    print(f"  {zone:10s}: {cnt:6,}건 ({pct:5.1f}%)")

print(f"\n[이벤트 감지]")
for ev in events:
    mask = (df.index >= ev['start']) & (df.index <= ev['end']) & df['operating']
    max_conc = df.loc[mask, 'dust_mg_m3'].max() if mask.any() else 0
    max_risk = df.loc[mask, 'risk_score'].max() if mask.any() else 0
    detected = "CUSUM 감지" if (df.loc[mask, 'alarm_upper']).any() else "미감지"
    print(f"  [{detected}] {ev['name']}")
    print(f"           기간: {ev['start'].strftime('%m/%d %H:%M')} ~ {ev['end'].strftime('%m/%d %H:%M')}")
    print(f"           최대 농도: {max_conc:.1f} mg/m3, 최대 위험점수: {max_risk:.3f}")

print(f"\n[TWA-8hr 준수 현황]")
op_df_summary = df[df['operating']].copy()
op_df_summary['date'] = op_df_summary.index.date
daily_avg = op_df_summary.groupby('date')['dust_mg_m3'].mean()
n_comply = (daily_avg <= TWA_LIMIT).sum()
n_violate = (daily_avg > TWA_LIMIT).sum()
print(f"  준수: {n_comply}일 / 위반: {n_violate}일")
if n_violate > 0:
    violate_dates = daily_avg[daily_avg > TWA_LIMIT]
    for d, v in violate_dates.items():
        print(f"    위반일: {d} (TWA={v:.2f} mg/m3)")

print("\n" + "=" * 70)
print("  결론: CUSUM 알고리즘은 점진적 농도 상승과 급격한 스파이크를")
print("  효과적으로 탐지하며, 복합 위험 점수와 결합하여 분진 폭발")
print("  사고를 사전에 예방할 수 있는 실용적 모니터링 도구입니다.")
print("=" * 70)